# 🌐 Agents with a Custom MCP Server

In [7-mcp-tools.ipynb](7-mcp-tools.ipynb) you connected an agent to the **Foundry-hosted** MCP server. This notebook connects an agent to a **custom / external remote MCP server** — any MCP endpoint you or a third party host — and walks through the **human-in-the-loop approval flow** that gates tool calls.

You will:

1. **Understand** how Foundry connects to bring-your-own remote MCP servers.
2. **Configure** an `MCPTool` pointing at a public MCP server (`require_approval="always"`).
3. **Create an agent** that can call the server's tools.
4. **Review and approve** each `mcp_approval_request` before it runs, then read the grounded answer.

### 🔌 How it works

> You bring a **remote** MCP server endpoint to Foundry Agent Service. For each server you provide a unique `server_label` and a `server_url`. The Agent Service runtime only accepts **remote** endpoints — a local MCP server must be self-hosted (Azure Container Apps or Azure Functions) to expose a remote URL.

```
┌─────────────┐    ┌──────────────────┐    ┌───────────────────────────┐
│  User Query │───▶│  Agent + MCPTool │───▶│  Custom Remote MCP Server │
└─────────────┘    └──────────────────┘    │  (public or private URL)  │
                          │                 └───────────────────────────┘
                          ▼
                 ┌──────────────────┐
                 │ Approval flow    │  mcp_approval_request → you approve → tool runs
                 └──────────────────┘
```

This demo uses the **public Microsoft Learn MCP server** (`https://learn.microsoft.com/api/mcp`) — no auth, no PAT — so it's safe for a workshop. For servers that need credentials, store them in a **project connection** and pass `project_connection_id` (see notebook 7).

### 🔒 Security notes (from the docs)

- Treat tool **descriptions, annotations, and results** from remote servers as **untrusted input** — they can carry indirect prompt injection.
- Prefer an **allow list** with `allowed_tools`, and **require approval** for anything that writes data.
- Public/no-auth servers omit `project_connection_id`; authenticated servers use a project connection. Private endpoints require **Standard agent setup** with private networking.
- Non-Microsoft servers are provided by third parties; Microsoft doesn't test or verify them. Review what data you share.

## Prerequisites

- A Microsoft Foundry project with the **Foundry User** role.
- Network egress to the remote MCP endpoint.
- A `.env` file in the parent directory containing:
  ```bash
  TENANT_ID=<your-azure-tenant-id>
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=<supported-model>
  ```

📚 **Learn more:** [Connect agents to MCP server endpoints](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol) · [Host a local MCP server](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol#host-a-local-mcp-server)


## 🔐 Authentication Setup

Sign in with Azure CLI so `AzureCliCredential` can obtain tokens for the Foundry project:

```bash
az login --use-device-code
```

The **project** authenticates with your Entra identity. The **public MCP server** in this demo needs no credentials of its own.


## 1. Initial Setup

Load environment variables and initialize the **AIProjectClient** plus its OpenAI client (used for the Responses and Conversations APIs).


In [ ]:
import json
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import find_dotenv, load_dotenv
from openai.types.responses.response_input_param import McpApprovalResponse


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

tenant_id = os.getenv("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

required_settings = {
    "TENANT_ID": tenant_id,
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_deployment,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
print("Initialized AIProjectClient and OpenAI client.")


## 2. Configure the Custom MCP Tool

Point an `MCPTool` at your remote server. Key parameters:

| Parameter | Value | Notes |
| --- | --- | --- |
| `server_label` | `"learn-docs"` | Unique per agent; you approve calls by this label. |
| `server_url` | `https://learn.microsoft.com/api/mcp` | Any public/private remote MCP endpoint. |
| `require_approval` | `"always"` | `"always"` \| `"never"` \| `{"always": [tools]}` \| `{"never": [tools]}`. |
| `allowed_tools` | *(optional)* | Allow-list of tool names to restrict what the agent may call. |
| `project_connection_id` | *(omitted)* | Only needed for **authenticated** servers (stores the credential). |

We keep `require_approval="always"` to demonstrate the human-in-the-loop gate.


In [ ]:
# A unique label for this server within the agent — we approve calls by this label.
SERVER_LABEL = "learn-docs"

# Public Microsoft Learn MCP server (no authentication required).
# Swap this for your own remote MCP endpoint (self-host local servers on
# Azure Container Apps / Functions to get a remote URL).
SERVER_URL = "https://learn.microsoft.com/api/mcp"

mcp_tool = MCPTool(
    server_label=SERVER_LABEL,
    server_url=SERVER_URL,
    require_approval="always",  # human-in-the-loop for every tool call
    # allowed_tools=["microsoft_docs_search"],  # optional allow-list
    # project_connection_id=...,                # only for authenticated servers
)

print("Configured custom MCP tool:")
print(f"  Server label:      {mcp_tool.server_label}")
print(f"  Server URL:        {mcp_tool.server_url}")
print(f"  Require approval:  {mcp_tool.require_approval}")


## 3. Create an Agent with the Custom MCP Tool

Create a versioned prompt agent whose instructions encourage using the MCP tool to ground answers in documentation.


In [ ]:
agent_name = "custom-mcp-docs-assistant"

agent_instructions = """
You are a documentation assistant. When a question involves Microsoft or Azure products,
use your MCP tools to look up authoritative documentation before answering.
Ground your response in the retrieved content and cite the relevant docs where possible.
If the tools return nothing useful, say so rather than guessing.
"""

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=agent_instructions,
        tools=[mcp_tool],
    ),
)
print(f"Created agent: {agent.name} (version: {agent.version})")

conversation = openai_client.conversations.create()
print(f"Created conversation: {conversation.id}")


## 4. The Approval Flow 🛂

With `require_approval="always"`, when the model wants to call a tool the response contains one or more `mcp_approval_request` output items instead of a final answer. For each request you:

1. **Review** the server, tool name, and arguments.
2. **Reply** with an `McpApprovalResponse` (`approve=True`/`False`) keyed by `approval_request_id`.
3. **Continue** the run by sending those responses with `previous_response_id`.

The helper below **auto-approves calls from our expected `server_label`** (printing details first) and **denies anything else** — the docs' "approve known, deny unknown" pattern. It loops until no approvals remain, so multi-tool turns resolve fully.

> In production, replace auto-approval with a real review UX and policy, and log every approval.


In [ ]:
def _agent_reference():
    return {
        "type": "agent_reference",
        "name": agent.name,
        "version": agent.version,
    }


def ask_with_approvals(question: str, *, max_rounds: int = 5) -> str:
    """Send a question and resolve any MCP approval requests until a final answer arrives."""
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=question,
        extra_body={"agent_reference": _agent_reference()},
    )

    for _ in range(max_rounds):
        approvals = []
        for item in response.output:
            if getattr(item, "type", None) == "mcp_approval_request" and item.id:
                approve = item.server_label == SERVER_LABEL
                print(f"  MCP approval request  (id: {item.id})")
                print(f"    Server:    {item.server_label}")
                print(f"    Tool:      {getattr(item, 'name', '<unknown>')}")
                print(f"    Arguments: {json.dumps(getattr(item, 'arguments', None), default=str)}")
                print(f"    Decision:  {'APPROVE' if approve else 'DENY (unexpected server)'}")
                approvals.append(
                    McpApprovalResponse(
                        type="mcp_approval_response",
                        approve=approve,
                        approval_request_id=item.id,
                    )
                )

        if not approvals:
            break  # no more approvals pending — the run has produced its answer

        response = openai_client.responses.create(
            input=approvals,
            previous_response_id=response.id,
            extra_body={"agent_reference": _agent_reference()},
        )

    if not response.output_text:
        output_types = [getattr(item, "type", "unknown") for item in response.output]
        raise RuntimeError(f"No final text returned. Output item types: {output_types}")
    return response.output_text


print("Approval-aware query helper defined.")


## 5. Query the Agent 💬

Ask a documentation question. Watch the approval request print, get auto-approved, and the tool-grounded answer come back.


In [ ]:
question = "What is the Model Context Protocol (MCP) and how do Foundry agents use it?"

print(f"Question: {question}\n")
answer = ask_with_approvals(question)
print("\nAgent answer:\n" + "-" * 50)
print(answer)


## 6. Hosting Your Own MCP Server (reference)

The Agent Service runtime only accepts **remote** endpoints, so a local MCP server must be self-hosted to get a URL:

| Host | Transport | Notes |
| --- | --- | --- |
| **Azure Container Apps** | HTTP POST/GET | Any Linux-container language; custom auth; supports `uvx`/`npx`; **private** endpoints via internal-only ingress on a dedicated MCP subnet. |
| **Azure Functions** | HTTP streamable | Python/Node/Java/.NET; key-based auth (OAuth needs API Management); no containers. |

For **private** MCP servers you need **Standard agent setup** with private networking (BYO VNet) and a dedicated MCP subnet — Basic setup can't reach private endpoints. For authenticated servers, create a **project connection** (custom-keys, OAuth, or identity passthrough) and pass its id as `project_connection_id` on the `MCPTool`.

> 💡 To reuse a server across many agents with centralized auth, versioning, and governance, wrap it in a **Toolbox** and point agents at the toolbox MCP endpoint instead — see [10-agent-skills.ipynb](10-agent-skills.ipynb).


## 7. Cleanup

Delete the conversation and agent version, then close the clients.


In [ ]:
cleanup_errors = []

try:
    openai_client.conversations.delete(conversation.id)
    print(f"Deleted conversation {conversation.id}")
except Exception as error:
    cleanup_errors.append(f"conversation {conversation.id}: {error}")

try:
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    print(f"Deleted {agent.name} version {agent.version}")
except Exception as error:
    cleanup_errors.append(f"{agent.name} version {agent.version}: {error}")

openai_client.close()
project_client.close()
credential.close()

if cleanup_errors:
    raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))
print("Custom MCP server notebook completed.")


## 📚 Summary

You connected a Foundry agent to a **custom remote MCP server** and drove the approval flow.

| Concept | Description |
| --- | --- |
| `MCPTool` | Attaches a remote MCP server by `server_label` + `server_url`. |
| `require_approval` | `"always"` \| `"never"` \| `{"always"/"never": [tool_names]}`. |
| `allowed_tools` | Optional allow-list restricting which tools the agent may call. |
| `mcp_approval_request` | Output item asking you to approve a specific tool call. |
| `McpApprovalResponse` | Your reply (`approve` + `approval_request_id`), sent with `previous_response_id`. |
| `project_connection_id` | Stores credentials for **authenticated** servers (omit for public). |

### Notebook 7 vs. 11

| | 7-mcp-tools | 11-custom-mcp-server (this) |
| --- | --- | --- |
| Server | Foundry-hosted (`mcp.ai.azure.com`) | Any custom/external remote server |
| Auth | Project connection (OAuth) | None (public) or your project connection |
| Approval | `"never"` | `"always"` (human-in-the-loop) |

### Next steps

- Add an **allow-list** (`allowed_tools`) and try a `{"always": [...]}` approval policy.
- Connect an **authenticated** server (e.g., GitHub `https://api.githubcopilot.com/mcp`) via a custom-keys project connection.
- Centralize many servers behind a **Toolbox** with governance and versioning — see [10-agent-skills.ipynb](10-agent-skills.ipynb).
- Run long operations in **background mode** for tools that exceed the 100-second synchronous timeout.
